# 🚢 Titanic — EDA (Exploratory Data Analysis)

**Propósito:** exploración inicial del dataset para informar las decisiones de feature engineering y arquitectura del modelo.

Este notebook es de **exploración** — el código de producción vive en `src/`.

**Preguntas que guían el análisis:**
1. ¿Qué variables tienen mayor correlación con la supervivencia?
2. ¿Qué nulos hay y cómo tratarlos?
3. ¿Hay desbalance de clases?
4. ¿Qué features derivados agregan valor predictivo?

In [ ]:
import sys
from pathlib import Path

# Añadir src/ al path para importar el paquete
sys.path.insert(0, str(Path().absolute().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from titanic_survival.data.loader import load_raw
from titanic_survival.features.engineering import feature_engineering

sns.set_theme(style='whitegrid', palette='husl', font_scale=1.1)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

print('✅ Imports OK')

## 1. Carga y resumen general

In [ ]:
df = load_raw('../data/raw/train.csv')
print(f'Shape: {df.shape}')
print(f'\nNulos:')
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_frame('nulos').assign(pct=lambda x: (x['nulos']/len(df)*100).round(1)))
print(f'\nBalance de clases:')
vc = df['Survived'].value_counts()
for k, v in vc.items():
    print(f'  {k}: {v} ({v/len(df):.1%})')
print(f'\n→ Desbalance: {vc[0]/vc[1]:.2f}x — usaremos pos_weight en BCEWithLogitsLoss')

## 2. Visualizaciones — Distribuciones y Supervivencia

In [ ]:
# GRÁFICA 1 — Target + supervivencia por las variables más importantes
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1a. Balance de clases
ax = axes[0, 0]
n_neg, n_pos = vc[0], vc[1]
bars = ax.bar(['No sobrevivió (0)', 'Sobrevivió (1)'], [n_neg, n_pos],
              color=['#E74C3C', '#2ECC71'], edgecolor='white')
ax.bar_label(bars, labels=[f'{n_neg}\n({n_neg/len(df):.1%})', f'{n_pos}\n({n_pos/len(df):.1%})'],
             padding=4, fontsize=11, fontweight='bold')
ax.set_title('Distribución del Target', fontweight='bold')
ax.set_ylim(0, 680)

# 1b. Supervivencia por Sexo
ax2 = axes[0, 1]
surv_sex = df.groupby('Sex')['Survived'].mean()
bars2 = ax2.bar(['Femenino', 'Masculino'],
                [surv_sex.get('female', 0), surv_sex.get('male', 0)],
                color=['#FF6B9D', '#4ECDC4'], edgecolor='white')
ax2.bar_label(bars2, fmt='{:.1%}', padding=3, fontsize=11)
ax2.axhline(df['Survived'].mean(), ls='--', color='gray', alpha=0.7,
            label=f'Promedio {df["Survived"].mean():.1%}')
ax2.set_title('Supervivencia por Sexo', fontweight='bold')
ax2.set_ylim(0, 0.9); ax2.legend()

# 1c. Supervivencia por Pclass
ax3 = axes[0, 2]
surv_class = df.groupby('Pclass')['Survived'].mean()
bars3 = ax3.bar(['1ª', '2ª', '3ª'], surv_class.values,
                color=['#F39C12', '#3498DB', '#95A5A6'], edgecolor='white')
ax3.bar_label(bars3, fmt='{:.1%}', padding=3, fontsize=11)
ax3.axhline(df['Survived'].mean(), ls='--', color='gray', alpha=0.7)
ax3.set_title('Supervivencia por Clase', fontweight='bold')
ax3.set_ylim(0, 0.8)

# 1d. Distribución de Edad
ax4 = axes[1, 0]
for s, lbl, c in [(0, 'No sobrevivió', '#E74C3C'), (1, 'Sobrevivió', '#2ECC71')]:
    data = df[df['Survived'] == s]['Age'].dropna()
    ax4.hist(data, bins=20, alpha=0.5, color=c, label=lbl, edgecolor='white')
    sns.kdeplot(data, ax=ax4, color=c, lw=2)
ax4.set_title(f'Distribución de Edad (177 nulos → mediana)', fontweight='bold')
ax4.set_xlabel('Edad'); ax4.legend(fontsize=9)

# 1e. Fare (log-transformado)
ax5 = axes[1, 1]
for s, lbl, c in [(0, 'No sobrevivió', '#E74C3C'), (1, 'Sobrevivió', '#2ECC71')]:
    data = np.log1p(df[df['Survived'] == s]['Fare'])
    ax5.hist(data, bins=20, alpha=0.5, color=c, label=lbl, edgecolor='white')
ax5.set_title('Distribución de log(1+Fare)', fontweight='bold')
ax5.set_xlabel('log(1 + Fare)'); ax5.legend(fontsize=9)
ax5.text(0.02, 0.96, '→ FareLog como feature', transform=ax5.transAxes,
         fontsize=9, va='top', color='gray')

# 1f. Heatmap Pclass × Sex
ax6 = axes[1, 2]
pivot = df.pivot_table(values='Survived', index='Pclass', columns='Sex', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn', ax=ax6,
            linewidths=0.5, linecolor='white')
ax6.set_title('Supervivencia: Clase × Sexo', fontweight='bold')
ax6.set_yticklabels(['1ª', '2ª', '3ª'], rotation=0)

plt.suptitle('EDA — Titanic: patrones de supervivencia', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Correlaciones con el target

In [ ]:
# GRÁFICA 2 — Correlaciones numéricas con Survived
num_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr = df[num_cols].corr()['Survived'].drop('Survived').sort_values(key=abs, ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = ['#E74C3C' if v < 0 else '#2ECC71' for v in corr]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Correlación con Survived\n(variables numéricas crudas)', fontweight='bold')
ax.set_xlabel('Correlación de Pearson')
for i, v in enumerate(corr.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), i, f'{v:.3f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)

# Heatmap completo
ax2 = axes[1]
corr_matrix = df[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', ax=ax2, center=0,
            linewidths=0.5, linecolor='white')
ax2.set_title('Matriz de Correlaciones', fontweight='bold')

plt.suptitle('Gráfica 2 — Correlaciones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Features engineered — Validación del valor predictivo

In [ ]:
# Aplicar feature engineering
df_fe = feature_engineering(df)

# GRÁFICA 3 — Supervivencia por features derivados
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 3a. Supervivencia por Título
ax = axes[0]
title_surv = df_fe.groupby('Title')['Survived'].agg(['mean', 'count']).reset_index()
title_surv.columns = ['Title', 'rate', 'count']
title_surv = title_surv.sort_values('rate', ascending=False)
ax.bar(title_surv['Title'], title_surv['rate'],
       color=sns.color_palette('husl', len(title_surv)), edgecolor='white')
ax.axhline(df['Survived'].mean(), ls='--', color='red', alpha=0.7)
ax.set_title('Supervivencia por Título\n(feature engineered)', fontweight='bold')
ax.set_ylabel('Tasa de supervivencia')
ax.set_ylim(0, 0.95)
ax.tick_params(axis='x', rotation=20)
for i, row in enumerate(title_surv.itertuples()):
    ax.text(i, row.rate + 0.01, f'{row.rate:.0%}\n(n={row.count})',
            ha='center', fontsize=8)

# 3b. Supervivencia por FamilySize
ax2 = axes[1]
fam_surv = df_fe.groupby('FamilySize')['Survived'].agg(['mean', 'count']).reset_index()
ax2.bar(fam_surv['FamilySize'].astype(str), fam_surv['mean'],
        color=sns.color_palette('husl', len(fam_surv)), edgecolor='white')
ax2.axhline(df['Survived'].mean(), ls='--', color='red', alpha=0.7)
ax2.set_title('Supervivencia por FamilySize\n(feature engineered)', fontweight='bold')
ax2.set_xlabel('FamilySize = SibSp + Parch + 1')
ax2.set_ylim(0, 0.8)

# 3c. Supervivencia por HasCabin
ax3 = axes[2]
cabin_surv = df_fe.groupby('HasCabin')['Survived'].agg(['mean', 'count']).reset_index()
bars3 = ax3.bar(['Sin cabina (0)', 'Con cabina (1)'], cabin_surv['mean'],
                color=['#E74C3C', '#2ECC71'], edgecolor='white')
ax3.bar_label(bars3, fmt='{:.1%}', padding=3, fontsize=12)
ax3.axhline(df['Survived'].mean(), ls='--', color='gray', alpha=0.7)
ax3.set_title('Supervivencia por HasCabin\n(proxy de clase alta)', fontweight='bold')
ax3.set_ylim(0, 0.75)

plt.suptitle('Gráfica 3 — Features Engineered vs Supervivencia', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Conclusiones del EDA

| Decisión | Justificación |
|---|---|
| **pos_weight = n_neg/n_pos × 2.0** | Desbalance 62%/38% → BCEWithLogitsLoss necesita compensación |
| **Age: imputar con mediana** | 19.9% nulos; distribución sesgada → mediana más robusta que media |
| **FareLog** | Distribución muy sesgada a la derecha; log1p normaliza |
| **Title** | Encapsula sexo + estatus + edad implícita → altamente predictivo |
| **FamilySize** | Relación no lineal: familias pequeñas sobreviven más que solos o grupos grandes |
| **HasCabin** | Proxy de clase alta → 30% más supervivencia con cabina conocida |
| **StratifiedKFold** | Preserva proporción 62%/38% en cada fold |
| **F1-Score como métrica principal** | Accuracy es engañosa con desbalance |